# Notebook 12 — Hyperparameter Re-Tuning

**Goal:** Re-tune LGBM and CatBoost-Plain on the 38-feature v2 set (NB10/NB11). Prior Optuna (NB05) ran on 29 features, fold 0 only. The new feature set shifts optimal hyperparameters.

**Why re-tune:**
- NB10/NB11 LGBM re-used NB05 params tuned on 29 features — 9 new features added since then
- CatBoost-Plain (0.9016) used default depth=8, lr=0.05 — never tuned
- reg_alpha=9.79 was tuned on fold 0 only — may be a local optimum for fold 0's race distribution

**Key changes from NB05 Optuna:**
1. Multi-fold objective: average AUC over **folds 0 and 3** (fold 0 alone produced fold-specific params)
2. Wider reg_alpha search: 1e-8 → **50.0** (previous 9.79 may be local optimum)
3. New LGBM params: **max_bin** (127–511) and **min_gain_to_split** (0–0.5)
4. n_trials = **200** for LGBM, **100** for CatBoost (each trial is slower)
5. CatBoost search space: depth (4–10), l2_leaf_reg (1–50), colsample_bylevel (0.4–1.0), min_data_in_leaf (5–200)

**Inputs:** `data/processed/train_features_v2.parquet`, `data/processed/test_features_v2.parquet`, `data/processed/fold_assignments.parquet`  
**Outputs:** `data/processed/oof_predictions_nb12.parquet`, `data/processed/test_predictions_nb12.parquet`, `models/nb12_summary.pkl`

**Success metric:** CatBoost re-tuned CV AUC > 0.906 OR LGBM re-tuned CV AUC > 0.880 → projected LB > 0.955.

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import time
import warnings
import pickle
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print('WARNING: catboost not installed — CatBoost cells will be skipped')

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
print(f'LightGBM  : {lgb.__version__}')
print(f'CatBoost  : {"available" if CATBOOST_AVAILABLE else "NOT installed"}')
print('Imports OK')

LightGBM  : 4.6.0
CatBoost  : available
Imports OK


In [3]:
# Robust project root detection — works from workspace root, notebooks/, or Kaggle
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root  = cwd
data_dir      = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

# Kaggle path fallback
if not data_dir.exists():
    data_dir      = Path('/kaggle/input/playground-series-s6e5')
    processed_dir = Path('/kaggle/working')

print(f'Project root : {project_root}')
print(f'Processed dir: {processed_dir}')

train = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test  = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds = pd.read_parquet(processed_dir / 'fold_assignments.parquet')

# Merge fold assignments into train (not stored in v2 parquet)
train = train.merge(folds, on='id', how='left')

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Fold distribution:\n{train["fold"].value_counts().sort_index().to_string()}')

Project root : c:\Repos\predict-f1-pit-stops
Processed dir: c:\Repos\predict-f1-pit-stops\data\processed
Train: (439140, 53)  |  Test: (188165, 51)
Fold distribution:
fold
0    88018
1    87444
2    88027
3    87854
4    87797


In [4]:
# ── Feature set: FEATS_T4 from NB10/NB11 (38 features) ──────────────────────
BASE_FEATURES = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Stint', 'Position',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
TARGET_ENC   = ['Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length']
TIER14_FEATS = [
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    'prime_pit_window', 'prime_window_x_compound',
    'abs_position_change', 'pos_change_rolling_std_3',
    'PitStop_lag1', 'laps_to_driver_end',
]
FEAT_COLS = BASE_FEATURES + TARGET_ENC + TIER14_FEATS  # 38 total

non_enc = BASE_FEATURES + TIER14_FEATS
assert not [f for f in non_enc if f not in train.columns], 'Missing train features'
assert not [f for f in non_enc if f not in test.columns],  'Missing test features'
print(f'Feature set: {len(FEAT_COLS)} features (26 base + 3 target enc + 9 Tier 1-4)')


def apply_target_encodings(train_df: pd.DataFrame, val_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute fold-aware target encodings from train_df, apply to val_df.
    Must be called INSIDE each CV fold — never on the full dataset.
    """
    global_mean  = train_df['PitNextLap'].mean()
    race_map     = train_df.groupby('Race')['PitNextLap'].mean()
    driver_map   = train_df.groupby('Driver')['PitNextLap'].mean()
    stint_map    = (
        train_df.groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max().reset_index().groupby('Driver')['TyreLife'].mean()
    )
    global_stint = stint_map.mean()

    out = val_df.copy()
    out['Race_target_encoded']     = out['Race'].map(race_map).fillna(global_mean)
    out['Driver_target_encoded']   = out['Driver'].map(driver_map).fillna(global_mean)
    out['Driver_avg_stint_length'] = out['Driver'].map(stint_map).fillna(global_stint)
    return out

print('apply_target_encodings defined.')

Feature set: 38 features (26 base + 3 target enc + 9 Tier 1-4)
apply_target_encodings defined.


## Generic CV Runner

Same `run_cv()` as NB11 — handles LGBM (with/without early stopping) and CatBoost.
Test predictions are fold-averaged across all 5 folds.

In [5]:
def run_cv(train_df, feature_cols, model_factory,
           test_df=None, lgbm_early_stopping=False, catboost_mode=False, label=''):
    """
    5-fold GroupKFold CV using pre-computed fold assignments.
    Applies fold-aware target encodings inside each fold.
    Returns: (oof_auc, fold_aucs, oof_preds, test_preds_avg)
    """
    n     = len(train_df)
    oof   = np.zeros(n)
    aucs  = []
    test_fold_preds = []
    t0    = time.time()

    for fold_idx in range(5):
        tr_mask  = train_df['fold'] != fold_idx
        val_mask = train_df['fold'] == fold_idx
        tr_idx   = np.where(tr_mask)[0]
        val_idx  = np.where(val_mask)[0]

        train_enc = apply_target_encodings(train_df.iloc[tr_idx], train_df.iloc[tr_idx])
        val_enc   = apply_target_encodings(train_df.iloc[tr_idx], train_df.iloc[val_idx])

        X_tr  = train_enc[feature_cols].to_numpy(dtype=np.float32)
        y_tr  = train_enc['PitNextLap'].to_numpy()
        X_val = val_enc[feature_cols].to_numpy(dtype=np.float32)
        y_val = val_enc['PitNextLap'].to_numpy()

        m = model_factory()

        if lgbm_early_stopping:
            callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)
        elif catboost_mode:
            m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
        else:
            m.fit(X_tr, y_tr)

        oof[val_idx] = m.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, oof[val_idx])
        aucs.append(fold_auc)

        if test_df is not None:
            test_enc = apply_target_encodings(train_df.iloc[tr_idx], test_df)
            X_test   = test_enc[feature_cols].to_numpy(dtype=np.float32)
            test_fold_preds.append(m.predict_proba(X_test)[:, 1])

        if lgbm_early_stopping:
            extra = f'trees={m.best_iteration_}'
        elif catboost_mode:
            extra = f'trees={m.best_iteration_}'
        else:
            extra = ''
        print(f'  Fold {fold_idx}: AUC={fold_auc:.4f}  {extra}')

    oof_auc = roc_auc_score(train_df['PitNextLap'], oof)
    elapsed = time.time() - t0
    print(f'  ─── OOF AUC: {oof_auc:.4f}  std={np.std(aucs):.4f}  n_feat={len(feature_cols)}  ({elapsed:.0f}s)')
    if label:
        print(f'  ─── {label}')

    test_avg = np.mean(test_fold_preds, axis=0) if test_fold_preds else None
    return oof_auc, np.array(aucs), oof, test_avg

print('run_cv defined.')

run_cv defined.


## Part A — LightGBM Re-Tuning

### Changes from NB05 Optuna

| Parameter | NB05 range | NB12 range | Reason |
|-----------|-----------|-----------|--------|
| Tune folds | fold 0 only | **folds 0 + 3** | Fold 0 alone produced fold-0-specific params |
| reg_alpha | 1e-8 → 10.0 | 1e-8 → **50.0** | 9.79 may be local optimum; allow more L1 |
| reg_lambda | 1e-8 → 10.0 | 1e-8 → **50.0** | Symmetric range with reg_alpha |
| num_leaves | 31 → 255 | 31 → **511** | 38 features may benefit from more capacity |
| min_child_samples | 10 → 100 | 5 → **200** | Wider range; new features change optimal leaf size |
| max_bin | — | **127 → 511** | Finer thresholds benefit compound-weighted features |
| min_gain_to_split | — | **0.0 → 0.5** | Alternative regularisation path vs reg_alpha |
| n_trials | 100 | **200** | Larger search space needs more exploration |

In [6]:
# Pre-compute train/val splits for folds 0 and 3 (used inside every trial)
# Computing inside the objective each trial would be redundant
TUNE_FOLDS = [0, 3]
tune_data  = {}  # fold_idx -> (X_tr, y_tr, X_val, y_val)

for tf in TUNE_FOLDS:
    tr_idx  = np.where(train['fold'] != tf)[0]
    val_idx = np.where(train['fold'] == tf)[0]
    train_enc = apply_target_encodings(train.iloc[tr_idx], train.iloc[tr_idx])
    val_enc   = apply_target_encodings(train.iloc[tr_idx], train.iloc[val_idx])
    tune_data[tf] = (
        train_enc[FEAT_COLS].to_numpy(dtype=np.float32),
        train_enc['PitNextLap'].to_numpy(),
        val_enc[FEAT_COLS].to_numpy(dtype=np.float32),
        val_enc['PitNextLap'].to_numpy(),
    )

print(f'Tune data prepared for folds: {TUNE_FOLDS}')
for tf, (X_tr, y_tr, X_val, y_val) in tune_data.items():
    print(f'  Fold {tf}: train={X_tr.shape}  val={X_val.shape}')


def lgbm_objective(trial):
    params = dict(
        n_estimators      = 2000,
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 31, 511),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 200),
        subsample         = trial.suggest_float('subsample', 0.4, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 50.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 50.0, log=True),
        max_bin           = trial.suggest_int('max_bin', 127, 511),
        min_gain_to_split = trial.suggest_float('min_gain_to_split', 0.0, 0.5),
        random_state      = 42,
        verbose           = -1,
    )
    aucs = []
    for tf in TUNE_FOLDS:
        X_tr, y_tr, X_val, y_val = tune_data[tf]
        m = lgb.LGBMClassifier(**params)
        callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)
        preds = m.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val, preds))
    return float(np.mean(aucs))


lgbm_study = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=42),
)

print('Starting LGBM Optuna (200 trials, folds 0+3) — ~20-40 min...')
t0 = time.time()
lgbm_study.optimize(lgbm_objective, n_trials=200, show_progress_bar=True)
print(f'\nBest avg AUC (folds 0+3): {lgbm_study.best_value:.4f}  ({time.time()-t0:.0f}s)')
print('Best LGBM params:')
for k, v in lgbm_study.best_params.items():
    print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')

Tune data prepared for folds: [0, 3]
  Fold 0: train=(351122, 38)  val=(88018, 38)
  Fold 3: train=(351286, 38)  val=(87854, 38)
Starting LGBM Optuna (200 trials, folds 0+3) — ~20-40 min...


Best trial: 181. Best value: 0.913561: 100%|██████████| 200/200 [2:31:41<00:00, 45.51s/it]  


Best avg AUC (folds 0+3): 0.9136  (9102s)
Best LGBM params:
  learning_rate: 0.022005
  num_leaves: 31
  min_child_samples: 12
  subsample: 0.967870
  colsample_bytree: 0.429823
  reg_alpha: 17.281674
  reg_lambda: 0.000018
  max_bin: 499
  min_gain_to_split: 0.369129


In [7]:
# ── LGBM: full 5-fold CV with re-tuned params ─────────────────────────────
LGBM_TUNED = dict(
    n_estimators = 2000,
    random_state = 42,
    verbose      = -1,
    **lgbm_study.best_params,
)

# Baseline: NB11 LGBM params (for delta comparison)
LGBM_BASELINE = dict(
    n_estimators=2000, learning_rate=0.049228, num_leaves=62,
    min_child_samples=91, subsample=0.752871, colsample_bytree=0.746388,
    reg_alpha=9.791678, reg_lambda=0.0, random_state=42, verbose=-1,
)

print('LGBM re-tuned — full 5-fold CV')
print(f'Params: {LGBM_TUNED}\n')

auc_lgbm_tuned, aucs_lgbm_tuned, oof_lgbm_tuned, test_lgbm_tuned = run_cv(
    train, FEAT_COLS,
    model_factory=lambda: lgb.LGBMClassifier(**LGBM_TUNED),
    test_df=test,
    lgbm_early_stopping=True,
    label='LGBM re-tuned (NB12 Optuna)',
)

LGBM_BASELINE_AUC = 0.8978  # NB11 reference
print(f'\n  Δ vs NB11 baseline (0.8978): {auc_lgbm_tuned - LGBM_BASELINE_AUC:+.4f}')

LGBM re-tuned — full 5-fold CV
Params: {'n_estimators': 2000, 'random_state': 42, 'verbose': -1, 'learning_rate': 0.022004860219019283, 'num_leaves': 31, 'min_child_samples': 12, 'subsample': 0.9678695939801807, 'colsample_bytree': 0.4298227704355011, 'reg_alpha': 17.28167401662238, 'reg_lambda': 1.8103877761997072e-05, 'max_bin': 499, 'min_gain_to_split': 0.3691288933102477}

  Fold 0: AUC=0.9099  trees=1028
  Fold 1: AUC=0.8821  trees=219
  Fold 2: AUC=0.9058  trees=314
  Fold 3: AUC=0.9172  trees=1498
  Fold 4: AUC=0.8968  trees=1482
  ─── OOF AUC: 0.9024  std=0.0121  n_feat=38  (97s)
  ─── LGBM re-tuned (NB12 Optuna)

  Δ vs NB11 baseline (0.8978): +0.0046


## Part B — CatBoost Re-Tuning

CatBoost-Plain achieved **0.9016 OOF AUC** in NB11 with default depth=8, lr=0.05 — never tuned. Key params to search:

| Parameter | Default | Search range | Reason |
|-----------|---------|-------------|--------|
| `depth` | 8 | 4–10 | Symmetric trees: 2^depth leaves per tree |
| `learning_rate` | 0.05 | 0.01–0.3 (log) | Interact with early stopping |
| `l2_leaf_reg` | 3.0 | 1–50 (log) | Regularises leaf values |
| `colsample_bylevel` | 1.0 | 0.4–1.0 | Feature subsampling per depth level |
| `min_data_in_leaf` | 1 | 5–200 | Leaf size regularisation |

Tune on folds 0+3 (same as LGBM), 100 trials. `boosting_type='Plain'` is fixed — the NB11 finding that ordered encoding fails with GroupKFold-by-Race is definitive.

In [8]:
if not CATBOOST_AVAILABLE:
    print('CatBoost not available — skipping Part B.')
    auc_cb_tuned = None
    oof_cb_tuned = None
    test_cb_tuned = None
    aucs_cb_tuned = None
else:
    def cb_objective(trial):
        params = dict(
            iterations        = 2000,
            learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            depth             = trial.suggest_int('depth', 4, 10),
            l2_leaf_reg       = trial.suggest_float('l2_leaf_reg', 1.0, 50.0, log=True),
            colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.4, 1.0),
            min_data_in_leaf  = trial.suggest_int('min_data_in_leaf', 5, 200),
            boosting_type     = 'Plain',
            eval_metric       = 'AUC',
            od_type           = 'Iter',
            od_wait           = 50,
            use_best_model    = True,
            verbose           = 0,
            random_seed       = 42,
        )
        aucs = []
        for tf in TUNE_FOLDS:
            X_tr, y_tr, X_val, y_val = tune_data[tf]
            m = CatBoostClassifier(**params)
            m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
            preds = m.predict_proba(X_val)[:, 1]
            aucs.append(roc_auc_score(y_val, preds))
        return float(np.mean(aucs))

    cb_study = optuna.create_study(
        direction = 'maximize',
        sampler   = optuna.samplers.TPESampler(seed=42),
    )

    print('Starting CatBoost Optuna (100 trials, folds 0+3) — ~10-20 min...')
    t0 = time.time()
    cb_study.optimize(cb_objective, n_trials=100, show_progress_bar=True)
    print(f'\nBest avg AUC (folds 0+3): {cb_study.best_value:.4f}  ({time.time()-t0:.0f}s)')
    print('Best CatBoost params:')
    for k, v in cb_study.best_params.items():
        print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')

Starting CatBoost Optuna (100 trials, folds 0+3) — ~10-20 min...


Best trial: 82. Best value: 0.912501: 100%|██████████| 100/100 [2:55:28<00:00, 105.29s/it]


Best avg AUC (folds 0+3): 0.9125  (10529s)
Best CatBoost params:
  learning_rate: 0.072425
  depth: 7
  l2_leaf_reg: 13.901203
  colsample_bylevel: 0.648597
  min_data_in_leaf: 72


In [9]:
if not CATBOOST_AVAILABLE:
    print('CatBoost not available — skipping.')
else:
    CB_TUNED = dict(
        iterations     = 2000,
        boosting_type  = 'Plain',
        eval_metric    = 'AUC',
        od_type        = 'Iter',
        od_wait        = 50,
        use_best_model = True,
        verbose        = 0,
        random_seed    = 42,
        **cb_study.best_params,
    )

    CB_BASELINE_AUC = 0.9016  # NB11 reference
    print('CatBoost re-tuned — full 5-fold CV')
    print(f'Params: {CB_TUNED}\n')

    auc_cb_tuned, aucs_cb_tuned, oof_cb_tuned, test_cb_tuned = run_cv(
        train, FEAT_COLS,
        model_factory=lambda: CatBoostClassifier(**CB_TUNED),
        test_df=test,
        catboost_mode=True,
        label='CatBoost re-tuned (NB12 Optuna)',
    )

    print(f'\n  Δ vs NB11 baseline (0.9016): {auc_cb_tuned - CB_BASELINE_AUC:+.4f}')

CatBoost re-tuned — full 5-fold CV
Params: {'iterations': 2000, 'boosting_type': 'Plain', 'eval_metric': 'AUC', 'od_type': 'Iter', 'od_wait': 50, 'use_best_model': True, 'verbose': 0, 'random_seed': 42, 'learning_rate': 0.0724250440220684, 'depth': 7, 'l2_leaf_reg': 13.901203365227097, 'colsample_bylevel': 0.6485971021910344, 'min_data_in_leaf': 72}

  Fold 0: AUC=0.9105  trees=662
  Fold 1: AUC=0.8895  trees=223
  Fold 2: AUC=0.8951  trees=0
  Fold 3: AUC=0.9146  trees=690
  Fold 4: AUC=0.8961  trees=703
  ─── OOF AUC: 0.8622  std=0.0096  n_feat=38  (176s)
  ─── CatBoost re-tuned (NB12 Optuna)

  Δ vs NB11 baseline (0.9016): -0.0394


## Comparison & Spearman ρ

Compare re-tuned models vs their NB11 baselines and compute correlation for NB13 ensemble planning.

In [10]:
# ── Summary table ────────────────────────────────────────────────────────────
print('OOF AUC — NB11 Baseline vs NB12 Re-tuned')
print('=' * 70)
print(f'  {"Model":<25} {"OOF AUC":>9}  {"Δ Baseline":>11}  Fold AUCs')
print('-' * 70)

rows = [
    ('lgbm_baseline (NB11)',  0.8978,       [0.9090, 0.8731, 0.9038, 0.9142, 0.8906]),
    ('lgbm_tuned   (NB12)',   auc_lgbm_tuned, aucs_lgbm_tuned.tolist()),
]
if auc_cb_tuned is not None:
    rows += [
        ('cb_baseline  (NB11)', 0.9016, [0.9081, 0.8864, 0.9047, 0.9126, 0.8945]),
        ('cb_tuned     (NB12)', auc_cb_tuned, aucs_cb_tuned.tolist()),
    ]

baseline_map = {'lgbm_tuned   (NB12)': 0.8978, 'cb_tuned     (NB12)': 0.9016}
for name, auc, fold_aucs in rows:
    delta = baseline_map.get(name.strip(), 0.0)
    delta_str = f'{auc - delta:+.4f}' if delta else '  ref  '
    fold_str = '  '.join(f'{a:.4f}' for a in fold_aucs)
    print(f'  {name:<25} {auc:>9.4f}  {delta_str:>11}  {fold_str}')

print()

# ── Spearman ρ between re-tuned models ──────────────────────────────────────
if auc_cb_tuned is not None:
    rho, _ = spearmanr(oof_lgbm_tuned, oof_cb_tuned)
    print(f'Spearman ρ (LGBM re-tuned × CatBoost re-tuned): {rho:.4f}')
    if rho < 0.90:
        print('  → ρ < 0.90 — meaningful diversity; stacking should help')
    elif rho < 0.95:
        print('  → 0.90 ≤ ρ < 0.95 — some diversity; stacking V2 pattern recommended')
    else:
        print('  → ρ ≥ 0.95 — high correlation; stacking gain will be small')

# ── Best model for NB13 ─────────────────────────────────────────────────────
print()
best_auc = auc_lgbm_tuned
best_name = 'lgbm_tuned'
if auc_cb_tuned is not None and auc_cb_tuned > best_auc:
    best_auc  = auc_cb_tuned
    best_name = 'cb_tuned'
print(f'Best single model for NB13: {best_name} — OOF AUC {best_auc:.4f}')
print(f'Projected LB: {best_auc:.4f} + 0.049 ≈ {best_auc + 0.049:.4f}')

OOF AUC — NB11 Baseline vs NB12 Re-tuned
  Model                       OOF AUC   Δ Baseline  Fold AUCs
----------------------------------------------------------------------
  lgbm_baseline (NB11)         0.8978        ref    0.9090  0.8731  0.9038  0.9142  0.8906
  lgbm_tuned   (NB12)          0.9024      +0.0046  0.9099  0.8821  0.9058  0.9172  0.8968
  cb_baseline  (NB11)          0.9016        ref    0.9081  0.8864  0.9047  0.9126  0.8945
  cb_tuned     (NB12)          0.8622      -0.0394  0.9105  0.8895  0.8951  0.9146  0.8961

Spearman ρ (LGBM re-tuned × CatBoost re-tuned): 0.8186
  → ρ < 0.90 — meaningful diversity; stacking should help

Best single model for NB13: lgbm_tuned — OOF AUC 0.9024
Projected LB: 0.9024 + 0.049 ≈ 0.9514


In [11]:
# ── Save OOF predictions ─────────────────────────────────────────────────────
oof_df = pd.DataFrame({'id': train['id'], 'fold': train['fold'], 'PitNextLap': train['PitNextLap']})
oof_df['lgbm_tuned_oof'] = oof_lgbm_tuned
if oof_cb_tuned is not None:
    oof_df['cb_tuned_oof'] = oof_cb_tuned

oof_path = processed_dir / 'oof_predictions_nb12.parquet'
oof_df.to_parquet(oof_path, index=False)
print(f'Saved OOF: {oof_path}  shape={oof_df.shape}')

# ── Save test predictions ────────────────────────────────────────────────────
test_df_out = pd.DataFrame({'id': test['id']})
if test_lgbm_tuned is not None:
    test_df_out['lgbm_tuned_pred'] = test_lgbm_tuned
if test_cb_tuned is not None:
    test_df_out['cb_tuned_pred'] = test_cb_tuned

test_path = processed_dir / 'test_predictions_nb12.parquet'
test_df_out.to_parquet(test_path, index=False)
print(f'Saved test: {test_path}  shape={test_df_out.shape}')

# ── Save summary pkl for NB13 ────────────────────────────────────────────────
summary = {
    'lgbm_tuned_params': LGBM_TUNED,
    'lgbm_tuned_auc': auc_lgbm_tuned,
    'lgbm_tuned_fold_aucs': aucs_lgbm_tuned,
    'lgbm_baseline_auc': 0.8978,
    'feature_cols': FEAT_COLS,
}
if CATBOOST_AVAILABLE and auc_cb_tuned is not None:
    summary.update({
        'cb_tuned_params': CB_TUNED,
        'cb_tuned_auc': auc_cb_tuned,
        'cb_tuned_fold_aucs': aucs_cb_tuned,
        'cb_baseline_auc': 0.9016,
        'spearman_rho_lgbm_cb': float(spearmanr(oof_lgbm_tuned, oof_cb_tuned)[0]),
    })

summary_path = models_dir / 'nb12_summary.pkl'
with open(summary_path, 'wb') as f:
    pickle.dump(summary, f)
print(f'Saved summary: {summary_path}')
print(f'Keys: {list(summary.keys())}')

Saved OOF: c:\Repos\predict-f1-pit-stops\data\processed\oof_predictions_nb12.parquet  shape=(439140, 5)
Saved test: c:\Repos\predict-f1-pit-stops\data\processed\test_predictions_nb12.parquet  shape=(188165, 3)
Saved summary: c:\Repos\predict-f1-pit-stops\models\nb12_summary.pkl
Keys: ['lgbm_tuned_params', 'lgbm_tuned_auc', 'lgbm_tuned_fold_aucs', 'lgbm_baseline_auc', 'feature_cols', 'cb_tuned_params', 'cb_tuned_auc', 'cb_tuned_fold_aucs', 'cb_baseline_auc', 'spearman_rho_lgbm_cb']


## Summary — Results and NB13 Strategy

### OOF AUC Results

| Model | OOF AUC | Δ Baseline | Fold AUCs | Status |
|-------|---------|-----------|-----------|--------|
| lgbm_baseline (NB11) | 0.8978 | — | 0.9090 / 0.8731 / 0.9038 / 0.9142 / 0.8906 | reference |
| **lgbm_tuned (NB12)** | **0.9024** | **+0.0046** | 0.9099 / 0.8821 / 0.9058 / 0.9172 / 0.8968 | **USE in NB13** |
| cb_baseline (NB11) | 0.9016 | — | 0.9081 / 0.8864 / 0.9047 / 0.9126 / 0.8945 | reference |
| ~~cb_tuned (NB12)~~ | ~~0.8622~~ | ~~−0.0394~~ | 0.9105 / 0.8895 / **0.8951 (trees=0)** / 0.9146 / 0.8961 | **INVALID — DO NOT USE** |

### LGBM Re-Tuning — Key Findings

**Multi-fold Optuna (200 trials, folds 0+3) found meaningfully different params than NB05 (fold 0 only):**

| Param | NB05 (fold 0) | NB12 (folds 0+3) | Direction |
|-------|--------------|-----------------|-----------|
| `num_leaves` | 62 | **31** | More regularized — reduces fold-specific overfitting |
| `reg_alpha` | 9.79 | **17.28** | Stronger L1 — confirms high sparsity preference |
| `max_bin` | — | **499** | Near-maximum — finer thresholds help compound features |
| `min_gain_to_split` | — | **0.369** | Additional split regularisation beyond reg_alpha |
| `learning_rate` | 0.049 | **0.022** | Slower — requires more trees (219–1498 vs 35–462) |

Fold 1 improved from 0.8731 → 0.8821 (+0.009) — the structural weak fold benefits from more regularization.

### CatBoost Re-Tuning — Failure Analysis

**Root cause: Fold 2 `best_iteration_=0` (1 tree used for predictions)**

The Optuna-tuned params (depth=7, lr=0.072, l2=13.9, min_data_in_leaf=72) are too aggressive for fold 2's race distribution. CatBoost determined that iteration 0 was the best and stopped, producing predictions compressed near the base rate (~0.2).

Within fold 2, AUC=0.8951 looks fine — rank ordering within a single fold is preserved even with near-flat predictions. But the **global OOF AUC** computes rank ordering across all 439k rows: fold 2's near-flat predictions are systematically misranked against folds 0/1/3/4's normal-scale predictions → global OOF 0.8622 is an artifact, not the model's real performance.

The ρ=0.8186 (LGBM NB12 × CatBoost NB12) is also artificial — created by fold 2's scale mismatch, not genuine model diversity.

**Fix for NB13:** Use CatBoost NB11 OOF predictions (0.9016, all folds correctly trained) instead of NB12 CatBoost.

### Notebook 13 Ensemble Plan

**Level-1 base models:**
- **Primary:** LGBM NB12 re-tuned — `oof_predictions_nb12.parquet → lgbm_tuned_oof` (0.9024)
- **Secondary:** CatBoost NB11 — `oof_predictions_nb11.parquet → cb_plain_oof` (0.9016)
- **Optional third:** ET NB11 — `oof_predictions_nb11.parquet → et_oof` (0.8788, ρ=0.931 vs LGBM)

First step in NB13: compute Spearman ρ between LGBM NB12 and CatBoost NB11 OOF. If ρ > 0.96, stacking may not help — submit LGBM NB12 solo first.

**Level-2 metalearner:** `LogisticRegression(C=0.05)` on:
```python
X_meta = np.column_stack([
    lgbm_tuned_platt, cb_plain_platt,          # base model scores (logit-transformed)
    Stint_scaled, prime_pit_window_scaled,      # top raw features (V2 stacking pattern)
    TyreLife_x_compound_ordinal_scaled, laps_to_driver_end_scaled,
])
```

**Test predictions to use:**
- LGBM: `test_predictions_nb12.parquet → lgbm_tuned_pred`
- CatBoost: `test_predictions_nb11.parquet → cb_plain_pred`

**Pre-check:** If metalearner OOF gain vs best solo (LGBM 0.9024) < +0.001, submit LGBM NB12 solo.  
**Projected LB:** LGBM solo 0.9024 + 0.049 ≈ **0.951**. With effective stacking: potentially 0.953–0.955.